In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from typing import TypedDict

In [2]:
model=ChatOllama(model='llama3.1:latest')

In [16]:
class BlogState(TypedDict):

    title:str
    outline:str
    content:str
    eval_score:int

In [4]:
def create_outline(state:BlogState)-> BlogState:

    title=state['title']

    prompt=f'Generate a detailed outline for a blog on the topic- {title}'
    outline=model.invoke(prompt).content

    state['outline']=outline

    return state

In [9]:
def create_blog(state:BlogState)->BlogState:

    title=state['title']
    outline=state['outline']

    prompt=f'Write a detailed blog on the title -{title} using the following outline \n {outline}'

    output=model.invoke(prompt).content

    state['content']=output
    return state

In [24]:
def evaluate(state:BlogState)->BlogState:
    title=state['title']
    outline=state['outline']
    blog=state['content']

    prompt=f'Based on outline Rate my blog out of 10.Note just Integer value.\n outline- {outline}\n\n blog- {blog}'

    score=model.invoke(prompt).content
    state['eval_score']=score

    return state

In [25]:
graph=StateGraph(BlogState)

graph.add_node("create_outline",create_outline)
graph.add_node("create_blog",create_blog)
graph.add_node("evaluate",evaluate)

graph.add_edge(START,"create_outline")
graph.add_edge("create_outline","create_blog")
graph.add_edge("create_blog","evaluate")
graph.add_edge("evaluate",END)

workflow=graph.compile()

In [26]:
intial_state={"title":"AI Ethics"}

output=workflow.invoke(intial_state)
output

{'title': 'AI Ethics',
 'outline': 'Here is a detailed outline for a blog on AI Ethics:\n\n**Title:** "Navigating the Complexities of AI Ethics: A Guide to Responsible AI Development and Deployment"\n\n**I. Introduction**\n\n* Brief overview of the importance of AI ethics\n* Explanation of the blog\'s purpose and scope\n* Thesis statement: As AI becomes increasingly integrated into our lives, it is essential that we prioritize AI ethics to ensure that these technologies are developed and deployed in a responsible and accountable manner.\n\n**II. The Need for AI Ethics**\n\n* Discussion of the benefits of AI (e.g., improved efficiency, enhanced decision-making)\n* Exploration of potential risks associated with AI (e.g., bias, job displacement, loss of human agency)\n* Explanation of why AI ethics is necessary to mitigate these risks and ensure that AI systems align with human values\n\n**III. Key Principles of AI Ethics**\n\n* Overview of the principles outlined in the "Asilomar AI Prin

In [27]:
output['title']

'AI Ethics'

In [28]:
output['outline']

'Here is a detailed outline for a blog on AI Ethics:\n\n**Title:** "Navigating the Complexities of AI Ethics: A Guide to Responsible AI Development and Deployment"\n\n**I. Introduction**\n\n* Brief overview of the importance of AI ethics\n* Explanation of the blog\'s purpose and scope\n* Thesis statement: As AI becomes increasingly integrated into our lives, it is essential that we prioritize AI ethics to ensure that these technologies are developed and deployed in a responsible and accountable manner.\n\n**II. The Need for AI Ethics**\n\n* Discussion of the benefits of AI (e.g., improved efficiency, enhanced decision-making)\n* Exploration of potential risks associated with AI (e.g., bias, job displacement, loss of human agency)\n* Explanation of why AI ethics is necessary to mitigate these risks and ensure that AI systems align with human values\n\n**III. Key Principles of AI Ethics**\n\n* Overview of the principles outlined in the "Asilomar AI Principles" (e.g., value alignment, fai

In [29]:
output['content']

'**Navigating the Complexities of AI Ethics: A Guide to Responsible AI Development and Deployment**\n\nAs artificial intelligence (AI) becomes increasingly integrated into our lives, it is essential that we prioritize AI ethics to ensure that these technologies are developed and deployed in a responsible and accountable manner. In this blog, we will explore the importance of AI ethics, discuss key principles and challenges, and provide best practices for integrating AI ethics into the development process.\n\n**The Need for AI Ethics**\n\nArtificial intelligence has the potential to bring about numerous benefits, including improved efficiency, enhanced decision-making, and increased productivity. However, as AI systems become more complex and widespread, there are growing concerns about their potential risks and consequences. Some of these risks include:\n\n*   **Bias and discrimination**: AI systems can perpetuate existing social biases, leading to discriminatory outcomes.\n*   **Job d

In [30]:
output['eval_score']

'I would rate this blog post out of 10 as follows:\n\n*   **Content (7/10)**: The blog post provides a clear and concise overview of AI ethics, covering key principles, challenges, and best practices for implementing AI ethics in the development process.\n*   **Organization and Structure (8/10)**: The blog post is well-organized, with each section clearly outlined and logically connected to the previous one. However, some sections could be expanded or rephrased for better clarity.\n*   **Writing Style (7/10)**: The writing style is clear and engaging, but occasionally may come across as overly formal or technical. Using simpler language and more conversational tone can make the content more accessible to a broader audience.\n*   **Depth of Information (6/10)**: While the blog post covers some essential aspects of AI ethics, it does not delve too deeply into specific examples or real-world case studies. Adding more in-depth analysis and concrete examples can strengthen the argument and 